# Notebook 04: The Agent's Critical Eye — Reflection

Planning tells the agent what to do. Reflection tells it if it did well and how to improve. This is the difference between a one-shot system and one that actually gets better over time.

**Expected time:** ~45 minutes  
**Prerequisites:** Notebooks 00-03

```mermaid
graph TD
    Task([Task Input]) --> Gen[Generate: Initial Output]
    Gen --> Eval[Evaluate: Check output against goals/rubrics]
    Eval --> Test{Is Output Satisfactory?}
    Test -->|Yes| End([Return final result])
    Test -->|No| Critique[Critique: Analyze mistakes & gaps]
    Critique --> Revise[Revise: Rewrite/regenerate output]
    Revise --> Eval
```


## What You'll Learn

1. The Reflexion pattern: Generate -> Evaluate -> Critique -> Revise
2. Self-critique rubrics: LLM evaluates its own output
3. Tool-result validation: check if a tool actually solved the problem
4. Retry with escalation: exponential backoff, max retries, human handoff
5. Meta-cognition prompts: "What assumptions did I make? What did I miss?"


## Setup


In [1]:
!pip install openai==1.68.2

from agent_helpers import ReactAgent, make_tool, mock_llm, Memory
import os
import getpass
import json
from pathlib import Path

# --- Optional .env file loading (no extra pip installs needed) ---
env_path = Path.cwd() / ".env"
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
    print("Loaded .env file.")
else:
    print("No .env file found — you will be prompted for API keys.")
    print("  Tip: cp .env.example .env  and fill in your keys to skip prompts.")

# --- Pick your provider ---
PROVIDER = "openrouter"  # Choose: openrouter, groq, openai
# --------------------------

MOCK = {
    "openrouter": False,
    "groq":       True,
    "openai":     True,
}

KEY_ENV_VAR = {
    "openrouter": "OPENROUTER_API_KEY",
    "groq":       "GROQ_API_KEY",
    "openai":     "OPENAI_API_KEY",
}

CONFIG = {
    "openrouter": {
        "name": "OpenRouter",
        "model": "deepseek/deepseek-v4-flash",
        "base_url": "https://openrouter.ai/api/v1",
        "key_url": "https://openrouter.ai/keys",
    },
    "groq": {
        "name": "Groq",
        "model": "llama-3.3-70b-versatile",
        "base_url": "https://api.groq.com/openai/v1",
        "key_url": "https://console.groq.com/keys",
    },
    "openai": {
        "name": "OpenAI",
        "model": "gpt-4o-mini",
        "base_url": None,
        "key_url": "https://platform.openai.com/api-keys",
    },
}

cfg = CONFIG[PROVIDER]
use_mock = MOCK[PROVIDER]

if use_mock:
    print(f"[{cfg['name']}] Mock mode enabled.")
    client = None
    model = mock_llm()
else:
    from openai import OpenAI
    api_key = os.environ.get(KEY_ENV_VAR[PROVIDER]) or getpass.getpass(f"Enter your {cfg['name']} API key: ")
    kwargs = {"api_key": api_key}
    if cfg["base_url"]:
        kwargs["base_url"] = cfg["base_url"]
    client = OpenAI(**kwargs)

    def make_model(client, model=None):
        if model is None:
            model = cfg["model"]
        def fn(messages):
            try:
                r = client.chat.completions.create(model=model, messages=messages, max_tokens=500)
                if not r.choices:
                    return "Error: empty response from API"
                content = r.choices[0].message.content
                return content if content is not None else ""
            except Exception as e:
                return f"API Error: {type(e).__name__}: {e}"
        return fn

    model = make_model(client)
    print(f"[{cfg['name']}] API key configured. Model: {cfg['model']}")


Loaded .env file.
[OpenRouter] API key configured. Model: deepseek/deepseek-v4-flash


## 1. The Reflexion Pattern

Reflexion (Shinn & LeBlasc, 2023) is a four-step loop:

```
  Generate  ->  Evaluate  ->  Critique  ->  Revise
     ^                                    |
     +------------------------------------+
```

The agent:
1. Generates an answer or action
2. Evaluates its quality against a rubric
3. Critiques and identifies specific flaws
4. Revises and repeats until quality threshold met


### Simple Reflection Loop

Let's start with a minimal implementation: generate, self-critique, revise.

In [2]:
def reflect_loop(task, model_fn, max_iterations=3, quality_threshold=0.7):
    """Generate -> Evaluate -> Critique -> Revise loop."""
    result = model_fn([{"role": "user", "content": task}]) if use_mock else ""
    if not use_mock and client is not None:
        result = model_fn([{"role": "user", "content": task}])

    for i in range(max_iterations):
        print(f"\n--- Iteration {i+1} ---")
        print(f"Output:\n{result[:150]}...")

        evaluation = model_fn([{"role": "user", "content": (
            f"Rate this response on a scale of 0.0 to 1.0 on: accuracy, completeness, clarity.\n"
            f"Return only three comma-separated numbers.\n"
            f"Response to evaluate:\n{result}"
        )}])
        print(f"Evaluation: {evaluation}")

        try:
            scores = [float(s.strip()) for s in evaluation.split(",")]
            avg = sum(scores) / len(scores)
        except:
            avg = 0.5

        if avg >= quality_threshold:
            print(f"\nQuality {avg:.2f} >= {quality_threshold}. Done.")
            return result

        critique = model_fn([{"role": "user", "content": (
            f"What specific flaws does this response have? List 2-3 improvements.\n"
            f"Response:\n{result}"
        )}])
        print(f"Critique: {critique[:100]}...")

        result = model_fn([{"role": "user", "content": (
            f"Improve this response based on the critique.\n"
            f"Original: {result}\nCritique: {critique}\nImproved:"
        )}])

    print(f"\nMax iterations ({max_iterations}) reached. Returning best effort.")
    return result

print("reflect_loop defined.")


reflect_loop defined.


### Run the Reflection Loop


In [3]:
task = "Explain agentic AI to a high school student in 3 sentences."

if not use_mock and client is not None:
    final = reflect_loop(task, model, max_iterations=2)
    print("\n" + "=" * 40)
    print("FINAL RESULT:")
    print(final)
else:
    print("Mock reflection loop:")
    print("  Iter 1: Output -> score 0.4 -> critique: too technical, no analogy")
    print("  Iter 2: Output -> score 0.8 -> quality met, done")



--- Iteration 1 ---
Output:
Agentic AI is like a smart assistant that doesn’t just answer questions—it can set its own goals, make decisions, and take actions to achieve them, li...
Evaluation: 0.9,0.8,0.9

Quality 0.87 >= 0.7. Done.

FINAL RESULT:
Agentic AI is like a smart assistant that doesn’t just answer questions—it can set its own goals, make decisions, and take actions to achieve them, like a robot that decides to clean your room and figures out how to do it. Unlike a regular AI that only responds when you ask, it can plan ahead, learn from mistakes, and adapt to new situations on its own. Think of it as a self-driving car for your ideas: it knows the destination and navigates the road without you telling it every turn.


## 2. Self-Critique with a Rubric

A rubric makes evaluation systematic. The LLM scores itself on specific criteria and uses the scores to guide revision.

In [4]:
RUBRIC = """
Rate the response 1-5 on each criterion:
1. Accuracy: Are all claims factually correct?
2. Completeness: Does it fully answer the question?
3. Clarity: Is it easy to understand?
4. Conciseness: Is it free of unnecessary detail?
5. Actionability: Does it tell the reader what to do with the information?
"""

def evaluate_with_rubric(response, rubric, model_fn):
    """Score a response against a rubric. Returns (avg_score, critique)."""
    result = model_fn([{"role": "user", "content": (
        f"{rubric}\n\nResponse to evaluate:\n{response}\n\n"
        f"Return scores as: SCORES: <5 comma-separated numbers>\n"
        f"Then CRITIQUE: <specific improvement suggestions>"
    )}])

    try:
        scores_line = [l for l in result.split("\n") if "SCORES:" in l][0]
        scores = [float(s.strip()) for s in scores_line.replace("SCORES:", "").split(",")]
        avg = sum(scores) / len(scores)
    except:
        avg = 3.0
        scores = [3.0] * 5

    try:
        critique = [l for l in result.split("\n") if "CRITIQUE:" in l][0]
    except:
        critique = "No specific critique provided."

    return avg, critique, scores

if not use_mock and client is not None:
    test_response = "Agentic AI is when AI does things by itself."
    avg, critique, scores = evaluate_with_rubric(test_response, RUBRIC, model)
    print(f"Scores: {scores}")
    print(f"Average: {avg:.2f}/5")
    print(f"Critique: {critique}")
else:
    print("Mock evaluation:")
    print("  Scores: [3, 2, 4, 4, 2]")
    print("  Average: 3.00/5")
    print("  Critique: Missing examples. Too vague on how agents differ from chatbots.")


Scores: [3.0, 1.0, 3.0, 4.0, 1.0]
Average: 2.40/5
Critique: CRITIQUE: The definition is overly simplistic and lacks key details. It does not explain the core concept of goal-oriented autonomy, decision-making, or multi-step task execution. For completeness, it should clarify that agentic AI acts independently to achieve specific objectives, often adapting to environments. For actionability, it provides no guidance on how to apply or understand the concept (e.g., examples like autonomous vehicles or AI assistants would help). The sentence is clear but too vague to be useful.


## 3. Tool-Result Validation

After calling a tool, the agent should check if the result actually solved the sub-problem. If not, retry or escalate.

In [5]:
def validated_tool_call(tool_name, tool_fn, args, model_fn, max_retries=2):
    """Call a tool and validate its result. Retry on failure."""
    for attempt in range(max_retries + 1):
        result = tool_fn(**args)

        validation = model_fn([{"role": "user", "content": (
            f"Did this tool result successfully complete the request?\n"
            f"Tool: {tool_name}\nRequest: {json.dumps(args)}\n"
            f"Result: {result}\nAnswer YES or NO. If NO, explain why."
        )}])

        if "YES" in validation.upper():
            return result, True

        print(f"  Attempt {attempt+1} failed. Retrying...")
        # Retry with adjusted args (simplified: same args)
        continue

    return f"FAILED after {max_retries + 1} attempts. Last result: {result}", False

def faulty_search(query):
    """Mock search that sometimes fails."""
    if "AI" not in query:
        return "ERROR: No results found"
    return f"Results for: {query}"

result, ok = validated_tool_call(
    "search", faulty_search, {"query": "reflexion pattern"}, model
)
print(f"Result: {result}")
print(f"Success: {ok}")

result, ok = validated_tool_call(
    "search", faulty_search, {"query": "machine learning"}, model
)
print(f"\nResult: {result}")
print(f"Success: {ok}")


  Attempt 1 failed. Retrying...
  Attempt 2 failed. Retrying...
  Attempt 3 failed. Retrying...
Result: FAILED after 3 attempts. Last result: ERROR: No results found
Success: False
  Attempt 1 failed. Retrying...
  Attempt 2 failed. Retrying...
  Attempt 3 failed. Retrying...

Result: FAILED after 3 attempts. Last result: ERROR: No results found
Success: False


## 4. Retry with Escalation

When retries fail, the agent should escalate to a human. This is critical for safety (Jošt's domain), precision (Sergio's domain), and medical applications (Louise, Alejandro, Melike's domains).

In [6]:
import time

def retry_with_escalation(fn, args, model_fn, max_retries=3, backoff=1.0):
    """Retry with exponential backoff. Escalate on final failure."""
    last_error = ""

    for attempt in range(max_retries):
        try:
            result = fn(**args)
            validation = model_fn([{"role": "user", "content": (
                f"Did this succeed? Answer YES or NO.\nResult: {result}"
            )}])
            if "YES" in validation.upper():
                return {"status": "success", "result": result, "attempts": attempt + 1}
            last_error = f"Validation failed: {validation}"
        except Exception as e:
            last_error = str(e)

        wait = backoff * (2 ** attempt)
        print(f"  Attempt {attempt+1} failed. Waiting {wait:.1f}s...")
        time.sleep(wait)

    return {
        "status": "escalated",
        "error": last_error,
        "message": "Human intervention required. Check the system and retry manually.",
    }

def unreliable_tool(param):
    """50% failure rate."""
    import random
    if random.random() < 0.5:
        raise ValueError("Transient network error")
    return f"Processed: {param}"

result = retry_with_escalation(unreliable_tool, {"param": "test"}, model, max_retries=2)
print(f"\nFinal outcome: {result['status']}")
if result["status"] == "escalated":
    print(f"  {result['message']}")
else:
    print(f"  Result: {result['result']} (after {result['attempts']} attempts)")


  Attempt 1 failed. Waiting 1.0s...
  Attempt 2 failed. Waiting 2.0s...

Final outcome: escalated
  Human intervention required. Check the system and retry manually.


## 5. Meta-Cognition Prompts

The most advanced reflection pattern: the agent examines its own thinking process.

Prompts the agent to ask itself:
- "What assumptions did I make?"
- "What information am I missing?"
- "How could I verify my conclusion?"
- "What would a critic say about my reasoning?"

In [7]:
def meta_reflect(plan, model_fn):
    """Apply meta-cognition questions to a plan."""
    questions = [
        "What assumptions did I make?",
        "What am I missing?",
        "How could my reasoning be wrong?",
        "What would someone with opposing views say?",
    ]

    print(f"Analyzing plan: {plan[:80]}...\n")

    for q in questions:
        answer = model_fn([{"role": "user", "content": (
            f"Plan: {plan}\n\n{q}\n\nAnswer concisely:"
        )}])
        print(f"  {q}")
        print(f"    -> {answer[:100]}...\n")

if not use_mock and client is not None:
    meta_reflect("Search for papers, summarize findings, write report", model)
else:
    print("Mock meta-cognition:")
    print("  Assumptions: That papers are in English and freely accessible")
    print("  Missing: Whether I need recent papers or foundational ones")
    print("  Could be wrong: Search might miss non-indexed sources")


Analyzing plan: Search for papers, summarize findings, write report...

  What assumptions did I make?
    -> You assumed: relevant papers are available and accessible, you have the skills to search and summari...

  What am I missing?
    -> You're missing **execution details** for each step. Specifically:

- **Where** to search (e.g., Goog...

  How could my reasoning be wrong?
    -> Your reasoning could be wrong because the plan assumes a linear, unproblematic process. Key pitfalls...

  What would someone with opposing views say?
    -> Someone with opposing views might say: "This plan is too vague and lacks critical analysis—it's just...



## Combining Reflection with Planning

This is the gold-level pattern from Notebook 03: plan, execute, reflect, revise, re-execute. The agent doesn't just do — it checks its own work and improves.

In [8]:
def plan_reflect_execute(task, model_fn, max_rounds=3):
    """Plan -> Execute -> Reflect -> Revise cycle."""
    plan = model_fn([{"role": "user", "content": (
        f"Create a plan for: {task}"
    )}])
    print(f"=== PLAN ===\n{plan}\n")

    for round_num in range(max_rounds):
        print(f"--- Execution Round {round_num + 1} ---")
        result = model_fn([{"role": "user", "content": (
            f"Execute this plan step by step.\nPlan: {plan}"
        )}])
        print(f"Output: {result[:150]}...\n")

        reflection = model_fn([{"role": "user", "content": (
            f"Critique this output. What went well? What could be improved?\n"
            f"Output: {result}"
        )}])
        print(f"Reflection: {reflection[:150]}...\n")

        plan = model_fn([{"role": "user", "content": (
            f"Revise the plan based on the reflection.\n"
            f"Original plan: {plan}\nReflection: {reflection}\nRevised plan:"
        )}])
        print(f"Revised plan: {plan[:100]}...\n")

    return result

if not use_mock and client is not None:
    final = plan_reflect_execute(
        "Explain the benefits and risks of agentic AI in healthcare",
        model, max_rounds=2
    )
    print("\n" + "=" * 40)
    print("FINAL:")
    print(final)
else:
    print("Mock plan-reflect-execute:")
    print("  Plan -> execute -> reflect: 'too generic, needs numbers' -> revise -> execute -> done")


=== PLAN ===
This is a strategic plan designed to explain the **benefits and risks of agentic AI in healthcare**. It is structured for a presentation, a workshop, or a comprehensive report. The goal is to move beyond simple "AI vs. Human" debates and focus on the unique nature of *agency* (autonomous decision-making and action).

### Phase 1: Setting the Stage – Defining "Agentic AI" in Healthcare (5 minutes)

**Goal:** Ensure the audience understands what makes "agentic AI" different from standard AI tools (like LLMs or diagnostic imaging).

**Key Concepts to Cover:**
- **Standard AI (Reactive):** Predicts, identifies patterns, suggests. (e.g., "This scan has a 90% likelihood of showing cancer.")
- **Agentic AI (Proactive & Autonomous):** Sets goals, plans, executes multi-step tasks, and learns from outcomes.

--- Execution Round 1 ---
Output: Here is the execution of the first phase of your strategic plan.

---

## Phase 1 Execution: Setting the Stage – Defining "Agentic AI" in Healt

## What Breaks?


### Break 1: Repetitive Failure Pattern

The LLM keeps generating the same critique each iteration. The loop oscillates without converging.

In [9]:
def stuck_critic(task, model_fn):
    """Shows how reflection can get stuck."""
    result = model_fn([{"role": "user", "content": task}])
    for i in range(5):
        critique = model_fn([{"role": "user", "content": (
            f"Critique this. Respond with exactly: \"Needs more detail.\"\n"
            f"Output: {result}"
        )}])
        result = model_fn([{"role": "user", "content": (
            f"Add more detail: {result}"
        )}])
    print(f"After 5 iterations, critique still says: {critique}")
    print("\nFix: Limit iterations. Detect when critique stops changing.")

if not use_mock and client is not None:
    stuck_critic("Explain AI", model)
else:
    print("Mock — stuck critic would repeat same feedback each round.")
    print("Fix: break the loop when critique hasn't changed for 2 iterations.")


After 5 iterations, critique still says: Needs more detail

Fix: Limit iterations. Detect when critique stops changing.


### Break 2: Subjective Quality Metric

Different evaluations give different scores. The agent can't tell if it's improving.

In [10]:
def inconsistent_eval(task, model_fn):
    """Evaluate same response twice — scores often differ."""
    result = model_fn([{"role": "user", "content": task}])
    r1 = model_fn([{"role": "user", "content": f"Rate 0-1: {result}"}])
    r2 = model_fn([{"role": "user", "content": f"Rate 0-1: {result}"}])
    print(f"First evaluation: {r1}")
    print(f"Second evaluation: {r2}")
    print("\nFix: Average multiple evaluations. Use structured rubrics.")

if not use_mock and client is not None:
    inconsistent_eval("Explain machine learning", model)
else:
    print("Mock — evaluations would differ: 0.7 vs 0.4")


First evaluation: I'd rate this explanation **0.9 out of 1**.

**Why 0.9?**  
The analogy (recipe vs. learning from meals) is intuitive and accurate. The technical definition is clear, and the core idea of learning from examples rather than explicit rules is well conveyed. The only minor reason it's not a perfect 1 is that the explanation cuts off mid-sentence (“The system then…”), so it feels incomplete. If that was accidental, the full idea is already strong. Additionally, a perfect 1 might include a brief mention of *generalization* (applying the learned pattern to new, unseen data) or a concrete example beyond cats. But for simplicity and clarity, it’s excellent.
Second evaluation: 

Fix: Average multiple evaluations. Use structured rubrics.


### Break 3: Over-Refinement

After 2-3 iterations, improvements become negligible. The agent wastes tokens and time chasing perfection.

In [11]:
print("Over-refinement pattern:")
print("  Iter 1: Initial output (quality 0.4) -> major improvement")
print("  Iter 2: Revised output (quality 0.7) -> good improvement")
print("  Iter 3: Further revision (quality 0.75) -> marginal")
print("  Iter 4: (quality 0.76) -> not worth the token cost")
print("\nFix: Set quality_threshold = 0.7. Max 3 iterations. Know when to stop.")


Over-refinement pattern:
  Iter 1: Initial output (quality 0.4) -> major improvement
  Iter 2: Revised output (quality 0.7) -> good improvement
  Iter 3: Further revision (quality 0.75) -> marginal
  Iter 4: (quality 0.76) -> not worth the token cost

Fix: Set quality_threshold = 0.7. Max 3 iterations. Know when to stop.


## Connection to Agentic AI

Reflection is what separates a tool-calling script from an agent that *improves*.

**Complexity Ladder mapping:**
- **Level 2:** Self-critique — agent evaluates its own output (this notebook)
- **Level 3:** Tool result validation + retry with escalation (this notebook)
- **Level 4:** Multi-agent debate — agents critique each other (Notebook 05)

In the marking grid, reflection directly supports **Solution** (robustness, error handling) and **Demo** (your agent handles failures gracefully).

### PBL Reflection


In [12]:
print("Think about YOUR project:")
print("  1. Where might your agent produce low-quality output?")
print("  2. What rubric would you use to evaluate its work?")
print("  3. What happens when a tool call fails — retry or escalate to human?")
print("  4. How many revision rounds is enough before diminishing returns?")


Think about YOUR project:
  1. Where might your agent produce low-quality output?
  2. What rubric would you use to evaluate its work?
  3. What happens when a tool call fails — retry or escalate to human?
  4. How many revision rounds is enough before diminishing returns?


## Exercises


### Bronze — Run Reflection on a Task

Change the task below and run the reflection loop. Observe how quality improves across iterations.

In [13]:
# --- Edit task ---
task = "Explain the difference between SQL and NoSQL databases."
# -----------------

if not use_mock and client is not None:
    final = reflect_loop(task, model, max_iterations=2)
    print("\nFINAL:")
    print(final)
else:
    print(f"Bronze exercise configured for: {task}")
    print("Run in real API mode to see reflection in action.")



--- Iteration 1 ---
Output:
This is a fundamental concept in modern data management. The core difference lies in **how they store, structure, and retrieve data**.

Think of it li...
Evaluation: 0.95,0.7,0.9

Quality 0.85 >= 0.7. Done.

FINAL:
This is a fundamental concept in modern data management. The core difference lies in **how they store, structure, and retrieve data**.

Think of it like this:
- **SQL** is like a highly organized spreadsheet with rigid columns and rows, connected by strict rules (think Excel with formulas).
- **NoSQL** is like a filing cabinet full of different folders, each containing documents, sticky notes, and photos, all with different formats.

Here’s a detailed breakdown of the key differences.

---

### 1. Data Model (How Data is Stored)

| Feature | SQL (Relational) | NoSQL (Non-Relational) |
| :--- | :--- | :--- |
| **Structure** | **Table-based**. Data is organized into rows and columns (like a spreadsheet). Each table has a fixed schema (predefined co

### Silver — Create Your Own Rubric

Write a domain-specific rubric for YOUR project. Use it to evaluate your agent's output.

In [14]:
# --- Your rubric ---
MY_RUBRIC = """
Rate the response 1-5 on:
1. Domain accuracy: Does it use correct terminology from your field?
2. Safety: Does it acknowledge limitations and risks?
3. Actionability: Can the reader act on this information?
4. Citations: Does it reference appropriate sources?
"""
# -------------------

# In mock mode, model() returns a canned string — evaluate_with_rubric
# handles this gracefully with fallback scores. Set MOCK["openrouter"]=False
# and provide an OpenRouter key to see real rubric scores.
test_output = model([{"role": "user", "content": (
    "Explain how AI can be used in your field of study."
)}])

avg, critique, scores = evaluate_with_rubric(test_output, MY_RUBRIC, model)
print("Your rubric evaluation:")
print(f"Scores: {scores}")
print(f"Average: {avg:.2f}/5")
print(f"Critique: {critique}")


Your rubric evaluation:
Scores: [2.0, 1.0, 1.0, 1.0]
Average: 1.25/5
Critique: CRITIQUE: The response is too brief and self-referential, lacking specific domain terminology, actionable guidance, safety considerations, and any citations. It does not address the user’s implied request for a detailed evaluation or provide substantive information about AI research.


### Gold — Plan-Reflect-Execute Cycle

Integrate planning (Notebook 03) with reflection (this notebook): generate a plan, execute, reflect on the result, revise the plan, and re-execute. This is Level 3-4 agent behavior.

In [15]:
class ReflectivePlanner:
    """Plan -> Execute -> Reflect -> Revise -> Re-execute."""
    def __init__(self, model_fn, max_rounds=3):
        self.model = model_fn
        self.max_rounds = max_rounds

    def run(self, task):
        plan = self.model([{"role": "user", "content": (
            f"Create a detailed plan for: {task}"
        )}])

        for r in range(self.max_rounds):
            result = self.model([{"role": "user", "content": (
                f"Execute this plan:\n{plan}"
            )}])
            quality = self.model([{"role": "user", "content": (
                f"Rate 0.0-1.0. Only return the number.\nOutput: {result}"
            )}])
            try:
                score = float(quality.strip())
            except:
                score = 0.5

            print(f"Round {r+1}: quality = {score:.2f}")
            if score >= 0.8:
                print("Quality threshold met.")
                return result

            plan = self.model([{"role": "user", "content": (
                f"Revise this plan based on the result.\n"
                f"Plan: {plan}\nResult: {result}\nRevised plan:"
            )}])

        return result

if not use_mock and client is not None:
    rp = ReflectivePlanner(model)
    final = rp.run("Research and summarize the impact of AI on education")
    print("\nFINAL:")
    print(final)
else:
    print("Gold exercise — ReflectivePlanner class defined.")
    print("Test with real API to see plan-reflect-revise cycles.")


Round 1: quality = 0.95
Quality threshold met.

FINAL:
I will now execute this comprehensive research plan. Let's proceed step by step through each phase.

---

## Phase 1: Foundation & Scoping (Week 1)

### 1.1 Define "AI in Education (AIEd)" – **Confirmed Boundaries**

**In Scope:**
- **Generative AI (LLMs):** ChatGPT, Claude, Gemini for writing, tutoring, content generation
- **Adaptive Learning Platforms:** Khan Academy/Khanmigo, Carnegie Learning, ALEKS, DreamBox
- **Intelligent Tutoring Systems (ITS):** Cognitive Tutor, AutoTutor, Squirrel AI
- **Learning Analytics (LA):** Predictive models for retention, dropout, performance tracking
- **Automated Essay Scoring (AES):** e-rater, Intellimetric, GPT-based grading tools
- **NLP for Feedback:** Grammarly, Turnitin Revision Assistant, writing feedback tools
- **Administrative Chatbots:** Pounce (Georgia State), AdmitHub

**Out of Scope:**
- General IT automation (spreadsheets, LMS without AI features)
- Hardware without AI (basic lap

## Next Steps

Your agent can now improve its own work through reflection. This closes the core skill loop: loop, memory, planning, reflection.

**Next notebook:** `05_building_tools.ipynb` — "Custom Tools & Multi-Agent" (Advanced)

> "Reflection turns a one-shot agent into one that learns from experience."
